# PharmaOpt | Reproducible integration pipeline

**CMS strict review edition:** Uses the verified 2019→2022, 2020→2023, 2021→2024, 2022→2025 historical CMS file. Outputs go into `outputs/cms_strict_review_v01/` so original results remain unchanged. The year lag is conservative, not a verification of publication dates.

**Purpose:** Reconstruct the historical v0.4 package-year panel from the original Notebook 06 definitions, integrate the 13 rule-based recall-risk categories from Notebook 10, and export validated, Tableau-friendly data. Optionally train and compare v0.4 and v0.5.

**Data grain:** One row per **package NDC × prediction year (2022–2025)** for the main model tables. Detailed recall tables and long-form category tables have different grains and must not be joined directly to the model table without aggregation.

**Safety:** Reads from MySQL using `SELECT` queries only. Does **not** run Notebook 05, change MySQL tables, overwrite Notebook 06, or store your database password. It writes new files to the `outputs/` folder.

**Run from:** `C:\Users\jpiza\PharmaOpt` (your project root). Keep `drug-ndc-0001-of-0001.json.zip`, `cms_strict_demand_features.csv`, and the Notebook 10 risk CSVs in their current locations.

**Methodological limitations:** Retrospective FDA NDC snapshot, recall `report_date` used as a provisional availability proxy, current recall descriptions may have been edited after the historical cutoff, and CMS utilization may have a reporting lag. **2025 is a previously inspected development holdout, not a pristine test set.** Scores from class-balanced logistic regression are ranking scores, **not calibrated probabilities**.

## A. Preconditions and database connection
Run the cells in order. A MySQL password prompt will appear. The preflight cell stops early if a required local source file is missing.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import gc
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\jpiza\PharmaOpt")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

required_files = {
    "FDA NDC ZIP": PROJECT_ROOT / "drug-ndc-0001-of-0001.json.zip",
    "CMS features": PROJECT_ROOT / "cms_strict_demand_features.csv",
    "Temporal risk features": OUTPUT_DIR / "recall_risk_features_package_year_2022_2025_v05.csv",
    "Recall classifications": OUTPUT_DIR / "recall_risk_signals_v02_integration.csv",
}

print("PHARMAOPT — PIPELINE PREFLIGHT")
print("Project root:", PROJECT_ROOT)
for label, path in required_files.items():
    print(f"{label}: {'FOUND' if path.is_file() else 'MISSING'} — {path}")
missing = [str(p) for p in required_files.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
print("Preflight passed. Existing source files will not be modified.")
# All outputs from this review version go into a NEW folder.
# The previous model results and dashboard exports stay untouched.
RESULTS_DIR = OUTPUT_DIR / "cms_strict_review_v01"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("NEW output folder:", RESULTS_DIR)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import mysql.connector
from getpass import getpass

_db_password = getpass("MySQL root password (not stored): ")
try:
    connection = mysql.connector.connect(
        host="127.0.0.1", port=3306, user="root",
        password=_db_password, database="pharmaopt"
    )
finally:
    del _db_password

print("Connected to PharmaOpt:", connection.is_connected())

In [ ]:
# ============================================================
# SQL QUERY HELPER
# ============================================================

def query_df(sql, params=None):
    """
    Run a SELECT query against PharmaOpt
    and return the result as a pandas DataFrame.
    """

    cursor = connection.cursor()

    cursor.execute(sql, params or ())

    rows = cursor.fetchall()

    columns = [
        description[0]
        for description in cursor.description
    ]

    cursor.close()

    return pd.DataFrame(
        rows,
        columns=columns
    )

print("SQL → pandas helper ready.")

In [ ]:
import pandas as pd
source_counts_df = query_df("""
SELECT 'drug_product' AS table_name, COUNT(*) AS row_count FROM drug_product
UNION ALL SELECT 'drug_package', COUNT(*) FROM drug_package
UNION ALL SELECT 'shortage_event', COUNT(*) FROM shortage_event
UNION ALL SELECT 'recall_event', COUNT(*) FROM recall_event
UNION ALL SELECT 'recall_package', COUNT(*) FROM recall_package
""")
source_counts = {
    str(row.table_name): int(row.row_count)
    for row in source_counts_df.itertuples(index=False)
}
print("MySQL source snapshot:", source_counts)

## B. Reconstruct the original v0.4 panel
The following transformations are copied from **Notebook 06**. We reuse its actual modeling definitions instead of approximating them. We deliberately omit the old training/evaluation cells; they are unnecessary to reconstruct the feature panel.

In [ ]:
# ============================================================
# LOAD NDC PRODUCT ACTIVITY DATES
# ============================================================

import zipfile
import json

ndc_zip = PROJECT_ROOT / "drug-ndc-0001-of-0001.json.zip"

with zipfile.ZipFile(ndc_zip, "r") as z:
    json_filename = z.namelist()[0]

    with z.open(json_filename) as f:
        ndc_raw = json.load(f)

ndc_activity_raw = pd.json_normalize(
    ndc_raw["results"]
)

# Show fields related to time / marketing activity
activity_columns = [
    column
    for column in ndc_activity_raw.columns
    if (
        "date" in column.lower()
        or "marketing" in column.lower()
        or "expiration" in column.lower()
    )
]

print("NDC ACTIVITY-RELATED FIELDS")
print("=" * 60)

for column in activity_columns:
    print(column)

In [ ]:
# ============================================================
# ANALYSIS 4 — NDC PRODUCT ACTIVITY DATE AUDIT
# ============================================================

activity_df = ndc_activity_raw[
    [
        "product_id",
        "product_ndc",
        "marketing_start_date",
        "marketing_end_date",
        "listing_expiration_date"
    ]
].copy()


# ------------------------------------------------------------
# CONVERT FDA YYYYMMDD DATES
# ------------------------------------------------------------

date_columns = [
    "marketing_start_date",
    "marketing_end_date",
    "listing_expiration_date"
]

for column in date_columns:
    activity_df[column] = pd.to_datetime(
        activity_df[column].astype("string"),
        format="%Y%m%d",
        errors="coerce"
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("NDC ACTIVITY DATE AUDIT")
print("=" * 65)

print("Total products:", f"{len(activity_df):,}")

for column in date_columns:

    valid = activity_df[column].notna().sum()
    missing = activity_df[column].isna().sum()

    print()
    print(column)
    print("-" * 40)
    print("Valid dates:", f"{valid:,}")
    print("Missing dates:", f"{missing:,}")

    if valid > 0:
        print(
            "Earliest:",
            activity_df[column].min().date()
        )
        print(
            "Latest:",
            activity_df[column].max().date()
        )

In [ ]:
# Memory cleanup after extracting the FDA marketing activity dates.
for name in ("ndc_raw", "ndc_activity_raw"):
    globals().pop(name, None)
gc.collect()

assert not activity_df["product_id"].duplicated().any(), (
    "FDA product_id values are duplicated; investigate before building the panel."
)
print("Unique FDA product IDs:", f"{len(activity_df):,}")

In [ ]:
package_population = query_df("""
SELECT
    pkg.package_ndc,
    pkg.product_id,
    dp.product_ndc,
    dp.generic_name,
    dp.brand_name,
    dp.labeler_name,
    dp.dosage_form,
    dp.marketing_category
FROM drug_package pkg
INNER JOIN drug_product dp
    ON pkg.product_id = dp.product_id;
""")

assert not package_population["package_ndc"].duplicated().any(), (
    "Package NDC is not unique in MySQL source population."
)
print("Source drug packages:", f"{len(package_population):,}")

### Original Notebook 06 Cell 18 — package-year panel
Uses the original active-product cohort rules, shortage labels, and prior-recall features. **The original recall features use `recall_initiation_date`; the new qualitative features use `report_date`.** These are not equivalent historical availability timestamps.

In [ ]:
# ============================================================
# PHARMAOPT — MULTI-YEAR MODELING DATASET
# 2022 → 2025
# ============================================================

model_years = [2022, 2023, 2024, 2025]

modeling_frames = []

print("BUILDING PHARMAOPT MODELING PANEL")
print("=" * 70)


for year in model_years:

    index_date = pd.Timestamp(f"{year}-01-01")
    horizon_end = pd.Timestamp(f"{year}-12-31")

    print(f"\nProcessing {year}...")


    # --------------------------------------------------------
    # 1. ACTIVE PACKAGE POPULATION
    # --------------------------------------------------------

    year_population = package_population.merge(
        activity_df[
            [
                "product_id",
                "marketing_start_date",
                "marketing_end_date",
                "listing_expiration_date"
            ]
        ],
        on="product_id",
        how="left"
    )


    active_mask = (
        year_population["marketing_start_date"].notna()
        &
        (
            year_population["marketing_start_date"]
            <= index_date
        )
        &
        (
            year_population["marketing_end_date"].isna()
            |
            (
                year_population["marketing_end_date"]
                >= index_date
            )
        )
    )


    year_df = (
        year_population[active_mask]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 2. SHORTAGE TARGET
    # --------------------------------------------------------

    shortage_query = """
    SELECT DISTINCT
        linked_package_ndc AS package_ndc

    FROM shortage_event

    WHERE linked_package_ndc IS NOT NULL
      AND initial_posting_date >= %s
      AND initial_posting_date < %s;
    """

    next_year_date = pd.Timestamp(
        f"{year + 1}-01-01"
    )

    year_shortages = query_df(
        shortage_query,
        (
            index_date.date(),
            next_year_date.date()
        )
    )


    shortage_set = set(
        year_shortages["package_ndc"]
        .dropna()
        .astype(str)
    )


    year_df["shortage_target"] = (
        year_df["package_ndc"]
        .astype(str)
        .isin(shortage_set)
        .astype(int)
    )


    # --------------------------------------------------------
    # 3. RECALL FEATURES KNOWN BEFORE INDEX DATE
    # --------------------------------------------------------

    recall_query = """
    SELECT

        rp.linked_package_ndc AS package_ndc,

        COUNT(
            DISTINCT re.recall_number
        ) AS prior_recall_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class I'
                THEN re.recall_number
            END
        ) AS prior_class_I_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class II'
                THEN re.recall_number
            END
        ) AS prior_class_II_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class III'
                THEN re.recall_number
            END
        ) AS prior_class_III_count,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_1y,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_2y,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_5y,

        MAX(
            re.recall_initiation_date
        ) AS most_recent_prior_recall

    FROM recall_package rp

    INNER JOIN recall_event re
        ON rp.recall_number = re.recall_number

    WHERE rp.linked_package_ndc IS NOT NULL
      AND re.recall_initiation_date < %s

    GROUP BY rp.linked_package_ndc;
    """


    date_1y = (
        index_date - pd.DateOffset(years=1)
    ).date()

    date_2y = (
        index_date - pd.DateOffset(years=2)
    ).date()

    date_5y = (
        index_date - pd.DateOffset(years=5)
    ).date()


    year_recalls = query_df(
        recall_query,
        (
            date_1y,
            date_2y,
            date_5y,
            index_date.date()
        )
    )


    # --------------------------------------------------------
    # 4. MERGE RECALL FEATURES
    # --------------------------------------------------------

    year_df = year_df.merge(
        year_recalls,
        on="package_ndc",
        how="left"
    )


    recall_columns = [
        "prior_recall_count",
        "prior_class_I_count",
        "prior_class_II_count",
        "prior_class_III_count",
        "recalls_prior_1y",
        "recalls_prior_2y",
        "recalls_prior_5y"
    ]


    year_df[recall_columns] = (
        year_df[recall_columns]
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # 5. PRODUCT AGE
    # --------------------------------------------------------

    year_df["product_age_days"] = (
        index_date
        -
        year_df["marketing_start_date"]
    ).dt.days


    # --------------------------------------------------------
    # 6. RECALL RECENCY
    # --------------------------------------------------------

    year_df["most_recent_prior_recall"] = (
        pd.to_datetime(
            year_df["most_recent_prior_recall"]
        )
    )


    year_df["days_since_last_recall"] = (
        index_date
        -
        year_df["most_recent_prior_recall"]
    ).dt.days


    # Binary indicator is useful when recency is missing
    year_df["has_prior_recall"] = (
        year_df["prior_recall_count"] > 0
    ).astype(int)


    # --------------------------------------------------------
    # 7. IDENTIFY PREDICTION YEAR
    # --------------------------------------------------------

    year_df["prediction_year"] = year


    # --------------------------------------------------------
    # 8. STORE YEAR
    # --------------------------------------------------------

    modeling_frames.append(year_df)


    print(
        f"Active packages: {len(year_df):,}"
    )

    print(
        f"Shortage positives: "
        f"{year_df['shortage_target'].sum():,}"
    )


# ============================================================
# COMBINE ALL YEARS
# ============================================================

pharmaopt_model_df = pd.concat(
    modeling_frames,
    ignore_index=True
)


print("\n" + "=" * 70)
print("MULTI-YEAR MODELING DATASET COMPLETE")
print("=" * 70)

print(
    "Total package-year observations:",
    f"{len(pharmaopt_model_df):,}"
)

print(
    "Total shortage-positive observations:",
    f"{pharmaopt_model_df['shortage_target'].sum():,}"
)

print(
    "Total non-shortage observations:",
    f"{(pharmaopt_model_df['shortage_target'] == 0).sum():,}"
)

In [ ]:
for name in ("modeling_frames", "year_population", "year_df", "package_population", "activity_df"):
    globals().pop(name, None)
gc.collect()
print("Original package-year panel:", len(pharmaopt_model_df))

### Original Notebook 06 Cells 25 and 28 — shortage and supply-pressure features
Both steps use SQL `SELECT` queries. They do not modify the database.

In [ ]:
# ============================================================
# PHARMAOPT v0.2
# HISTORICAL SHORTAGE FEATURE ENGINEERING
# ============================================================

enhanced_frames = []

print("BUILDING HISTORICAL SHORTAGE FEATURES")
print("=" * 70)


for year in [2022, 2023, 2024, 2025]:

    print(f"\nProcessing historical features for {year}...")

    index_date = pd.Timestamp(f"{year}-01-01")

    # Get this year's existing modeling rows
    year_df = pharmaopt_model_df[
        pharmaopt_model_df["prediction_year"] == year
    ].copy()


    # --------------------------------------------------------
    # HISTORICAL SHORTAGES BEFORE PREDICTION DATE
    # --------------------------------------------------------

    prior_shortages = query_df("""
    SELECT

        s.linked_package_ndc AS package_ndc,

        dp.product_ndc,
        dp.generic_name,
        dp.labeler_name,

        s.initial_posting_date

    FROM shortage_event s

    INNER JOIN drug_package pkg
        ON s.linked_package_ndc = pkg.package_ndc

    INNER JOIN drug_product dp
        ON pkg.product_id = dp.product_id

    WHERE s.linked_package_ndc IS NOT NULL
      AND s.initial_posting_date IS NOT NULL
      AND s.initial_posting_date < %s;
    """, (index_date.date(),))


    # --------------------------------------------------------
    # PACKAGE-LEVEL PRIOR SHORTAGES
    # --------------------------------------------------------

    package_history = (
        prior_shortages
        .groupby("package_ndc")
        .size()
        .to_dict()
    )

    year_df["package_prior_shortages"] = (
        year_df["package_ndc"]
        .map(package_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # PRODUCT-NDC PRIOR SHORTAGES
    # --------------------------------------------------------

    product_history = (
        prior_shortages
        .groupby("product_ndc")
        .size()
        .to_dict()
    )

    year_df["product_prior_shortages"] = (
        year_df["product_ndc"]
        .map(product_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # GENERIC-DRUG PRIOR SHORTAGES
    # --------------------------------------------------------

    generic_history = (
        prior_shortages
        .groupby("generic_name")
        .size()
        .to_dict()
    )

    year_df["generic_prior_shortages"] = (
        year_df["generic_name"]
        .map(generic_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # LABELER / MANUFACTURER PRIOR SHORTAGES
    # --------------------------------------------------------

    labeler_history = (
        prior_shortages
        .groupby("labeler_name")
        .size()
        .to_dict()
    )

    year_df["labeler_prior_shortages"] = (
        year_df["labeler_name"]
        .map(labeler_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # RECENT SHORTAGES — PRIOR 365 DAYS
    # --------------------------------------------------------

    one_year_before = (
        index_date - pd.DateOffset(years=1)
    )

    recent_shortages = prior_shortages[
        pd.to_datetime(
            prior_shortages["initial_posting_date"]
        ) >= one_year_before
    ].copy()


    generic_recent = (
        recent_shortages
        .groupby("generic_name")
        .size()
        .to_dict()
    )

    labeler_recent = (
        recent_shortages
        .groupby("labeler_name")
        .size()
        .to_dict()
    )


    year_df["generic_shortages_prior_1y"] = (
        year_df["generic_name"]
        .map(generic_recent)
        .fillna(0)
        .astype(int)
    )


    year_df["labeler_shortages_prior_1y"] = (
        year_df["labeler_name"]
        .map(labeler_recent)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # SIMPLE BINARY HISTORY FLAGS
    # --------------------------------------------------------

    year_df["package_has_shortage_history"] = (
        year_df["package_prior_shortages"] > 0
    ).astype(int)

    year_df["product_has_shortage_history"] = (
        year_df["product_prior_shortages"] > 0
    ).astype(int)

    year_df["generic_has_shortage_history"] = (
        year_df["generic_prior_shortages"] > 0
    ).astype(int)

    year_df["labeler_has_shortage_history"] = (
        year_df["labeler_prior_shortages"] > 0
    ).astype(int)


    enhanced_frames.append(year_df)


# ============================================================
# COMBINE ENHANCED PANEL
# ============================================================

pharmaopt_v02_df = pd.concat(
    enhanced_frames,
    ignore_index=True
)


print("\n" + "=" * 70)
print("PHARMAOPT v0.2 FEATURE TABLE COMPLETE")
print("=" * 70)

print(
    "Observations:",
    f"{len(pharmaopt_v02_df):,}"
)

print(
    "Features added:",
    10
)

print()

print(
    pharmaopt_v02_df[
        [
            "package_ndc",
            "prediction_year",
            "package_prior_shortages",
            "product_prior_shortages",
            "generic_prior_shortages",
            "labeler_prior_shortages",
            "shortage_target"
        ]
    ].head(10)
)

In [ ]:
for name in ("enhanced_frames", "pharmaopt_model_df", "prior_shortages"):
    globals().pop(name, None)
gc.collect()

In [ ]:
# ============================================================
# PHARMAOPT v0.3
# SUPPLY-PRESSURE + RECENCY FEATURES
# ============================================================

import numpy as np

v03_frames = []

print("BUILDING PHARMAOPT v0.3 FEATURES")
print("=" * 75)


for year in [2022, 2023, 2024, 2025]:

    print(f"\nProcessing {year}...")

    index_date = pd.Timestamp(f"{year}-01-01")

    year_df = pharmaopt_v02_df[
        pharmaopt_v02_df["prediction_year"] == year
    ].copy()


    # ========================================================
    # 1. CURRENT ACTIVE PRODUCT/MARKET BREADTH
    # ========================================================

    product_breadth = (
        year_df.groupby("product_ndc")["package_ndc"]
        .nunique()
        .to_dict()
    )

    generic_breadth = (
        year_df.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_breadth = (
        year_df.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["product_active_packages"] = (
        year_df["product_ndc"]
        .map(product_breadth)
        .fillna(1)
    )

    year_df["generic_active_packages"] = (
        year_df["generic_name"]
        .map(generic_breadth)
        .fillna(1)
    )

    year_df["labeler_active_packages"] = (
        year_df["labeler_name"]
        .map(labeler_breadth)
        .fillna(1)
    )


    # ========================================================
    # 2. PRIOR SHORTAGE EVENTS
    # ========================================================

    history = query_df("""
        SELECT
            s.linked_package_ndc AS package_ndc,
            dp.product_ndc,
            dp.generic_name,
            dp.labeler_name,
            s.initial_posting_date

        FROM shortage_event s

        INNER JOIN drug_package pkg
            ON s.linked_package_ndc = pkg.package_ndc

        INNER JOIN drug_product dp
            ON pkg.product_id = dp.product_id

        WHERE s.linked_package_ndc IS NOT NULL
          AND s.initial_posting_date IS NOT NULL
          AND s.initial_posting_date < %s;
    """, (index_date.date(),))


    history["initial_posting_date"] = pd.to_datetime(
        history["initial_posting_date"]
    )


    # ========================================================
    # 3. LAST SHORTAGE DATE / RECENCY
    # ========================================================

    package_last = (
        history.groupby("package_ndc")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    product_last = (
        history.groupby("product_ndc")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    generic_last = (
        history.groupby("generic_name")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    labeler_last = (
        history.groupby("labeler_name")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )


    year_df["package_last_shortage"] = (
        year_df["package_ndc"].map(package_last)
    )

    year_df["product_last_shortage"] = (
        year_df["product_ndc"].map(product_last)
    )

    year_df["generic_last_shortage"] = (
        year_df["generic_name"].map(generic_last)
    )

    year_df["labeler_last_shortage"] = (
        year_df["labeler_name"].map(labeler_last)
    )


    for level in [
        "package",
        "product",
        "generic",
        "labeler"
    ]:

        year_df[f"{level}_days_since_shortage"] = (
            index_date
            -
            pd.to_datetime(
                year_df[f"{level}_last_shortage"]
            )
        ).dt.days


    # ========================================================
    # 4. DISTINCT PACKAGES EVER AFFECTED
    # ========================================================

    product_affected = (
        history.groupby("product_ndc")["package_ndc"]
        .nunique()
        .to_dict()
    )

    generic_affected = (
        history.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_affected = (
        history.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["product_prior_affected_packages"] = (
        year_df["product_ndc"]
        .map(product_affected)
        .fillna(0)
    )

    year_df["generic_prior_affected_packages"] = (
        year_df["generic_name"]
        .map(generic_affected)
        .fillna(0)
    )

    year_df["labeler_prior_affected_packages"] = (
        year_df["labeler_name"]
        .map(labeler_affected)
        .fillna(0)
    )


    # ========================================================
    # 5. RECENT SHORTAGE HISTORY — PRIOR 365 DAYS
    # ========================================================

    one_year_before = (
        index_date - pd.DateOffset(years=1)
    )

    recent = history[
        history["initial_posting_date"] >= one_year_before
    ].copy()


    package_recent = (
        recent.groupby("package_ndc")
        .size()
        .to_dict()
    )

    product_recent = (
        recent.groupby("product_ndc")
        .size()
        .to_dict()
    )

    generic_recent_affected = (
        recent.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_recent_affected = (
        recent.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["package_shortages_prior_1y"] = (
        year_df["package_ndc"]
        .map(package_recent)
        .fillna(0)
    )

    year_df["product_shortages_prior_1y"] = (
        year_df["product_ndc"]
        .map(product_recent)
        .fillna(0)
    )

    year_df["generic_recent_affected_packages"] = (
        year_df["generic_name"]
        .map(generic_recent_affected)
        .fillna(0)
    )

    year_df["labeler_recent_affected_packages"] = (
        year_df["labeler_name"]
        .map(labeler_recent_affected)
        .fillna(0)
    )


    # ========================================================
    # 6. NORMALIZED SUPPLY-PRESSURE FEATURES
    # ========================================================

    year_df["product_shortage_pressure"] = (
        year_df["product_prior_affected_packages"]
        /
        year_df["product_active_packages"]
    )

    year_df["generic_shortage_pressure"] = (
        year_df["generic_prior_affected_packages"]
        /
        year_df["generic_active_packages"]
    )

    year_df["labeler_shortage_pressure"] = (
        year_df["labeler_prior_affected_packages"]
        /
        year_df["labeler_active_packages"]
    )


    # ========================================================
    # 7. SIBLING PACKAGE SIGNAL
    # ========================================================

    year_df["product_sibling_shortage_packages"] = (
        year_df["product_prior_affected_packages"]
        -
        year_df["package_has_shortage_history"]
    ).clip(lower=0)


    year_df["generic_sibling_shortage_packages"] = (
        year_df["generic_prior_affected_packages"]
        -
        year_df["package_has_shortage_history"]
    ).clip(lower=0)


    # ========================================================
    # 8. LOG-TRANSFORM LARGE COUNTS
    # ========================================================

    year_df["log_labeler_prior_shortages"] = np.log1p(
        year_df["labeler_prior_shortages"]
    )

    year_df["log_generic_prior_shortages"] = np.log1p(
        year_df["generic_prior_shortages"]
    )


    v03_frames.append(year_df)


# ============================================================
# COMBINE
# ============================================================

pharmaopt_v03_df = pd.concat(
    v03_frames,
    ignore_index=True
)


print("\n" + "=" * 75)
print("PHARMAOPT v0.3 FEATURE TABLE COMPLETE")
print("=" * 75)

print(
    "Observations:",
    f"{len(pharmaopt_v03_df):,}"
)

print(
    "Shortage positives:",
    f"{pharmaopt_v03_df['shortage_target'].sum():,}"
)

In [ ]:
for name in ("v03_frames", "pharmaopt_v02_df", "history"):
    globals().pop(name, None)
gc.collect()

### Original Notebook 06 Cells 32 and 33 — CMS Part D join
The merge remains `many_to_one` on normalized generic name and prediction year, but uses the existing **strict historical** CMS features. The code checks the source year mapping and the merged panel.

In [ ]:
# ============================================================
# PHARMAOPT v0.4
# IMPORT CMS DEMAND FEATURES
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

PROJECT_DIR = PROJECT_ROOT

cms_features = pd.read_csv(
    PROJECT_DIR / "cms_strict_demand_features.csv"
)

# Fail early if the historical file is incomplete or misaligned.
expected_cms_year_map = {2019: 2022, 2020: 2023, 2021: 2024, 2022: 2025}
required_cms_cols = {
    "generic_normalized", "cms_data_year", "prediction_year",
    "cms_total_claims", "cms_30day_fills", "cms_total_drug_cost",
    "cms_brand_count", "cms_claim_growth", "cms_fill_growth",
    "log_cms_claims", "log_cms_fills", "log_cms_cost"
}
missing_cms_cols = sorted(required_cms_cols - set(cms_features.columns))
if missing_cms_cols:
    raise ValueError(f"Strict CMS file is missing columns: {missing_cms_cols}")
if cms_features["generic_normalized"].isna().any():
    raise ValueError("Strict CMS file has missing generic names")
if cms_features[["generic_normalized", "prediction_year"]].duplicated().any():
    raise ValueError("Strict CMS file has duplicate generic-year merge keys")
actual_cms_pairs = set(zip(cms_features["cms_data_year"], cms_features["prediction_year"]))
if actual_cms_pairs != set(expected_cms_year_map.items()):
    raise ValueError(f"Incorrect or incomplete CMS year mapping: {sorted(actual_cms_pairs)}")
print("STRICT CMS FILE AUDIT PASSED")
print("CMS feature rows:", f"{len(cms_features):,}")
print("CMS → prediction years:", sorted(actual_cms_pairs))
print()

print("CMS prediction years:")
print(
    cms_features["prediction_year"]
    .value_counts()
    .sort_index()
)

display(cms_features.head())

In [ ]:
# ============================================================
# LINK CMS DEMAND → PHARMAOPT PACKAGE-YEAR PANEL
# ============================================================

def normalize_drug_name(value):

    if pd.isna(value):
        return None

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = value.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


pharmaopt_v04_df = pharmaopt_v03_df.copy()

pharmaopt_v04_df["generic_normalized"] = (
    pharmaopt_v04_df["generic_name"]
    .apply(normalize_drug_name)
)


cms_merge_columns = [
    "generic_normalized",
    "prediction_year",
    "cms_data_year",
    "cms_total_claims",
    "cms_30day_fills",
    "cms_total_drug_cost",
    "cms_brand_count",
    "cms_claim_growth",
    "cms_fill_growth",
    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost"
]


pharmaopt_v04_df = pharmaopt_v04_df.merge(
    cms_features[cms_merge_columns],
    on=[
        "generic_normalized",
        "prediction_year"
    ],
    how="left",
    validate="many_to_one"
)


pharmaopt_v04_df["cms_demand_match"] = (
    pharmaopt_v04_df["cms_total_claims"]
    .notna()
    .astype(int)
)


# Check the ACTUAL merged package-year rows, not just the source CSV.
matched_cms_rows = pharmaopt_v04_df["cms_demand_match"].eq(1)
if not matched_cms_rows.any():
    raise ValueError("No CMS demand features matched the package-year panel")
if not (
    pharmaopt_v04_df.loc[matched_cms_rows, "prediction_year"]
    - pharmaopt_v04_df.loc[matched_cms_rows, "cms_data_year"]
).eq(3).all():
    raise ValueError("CMS historical timing failed in the merged panel")
print("STRICT CMS JOIN TIMING PASSED")
print("PHARMAOPT v0.4 MERGE COMPLETE")
print("=" * 75)

print(
    "Package-year observations:",
    f"{len(pharmaopt_v04_df):,}"
)

print()

for year in [2022, 2023, 2024, 2025]:

    temp = pharmaopt_v04_df[
        pharmaopt_v04_df["prediction_year"] == year
    ]

    matched = int(
        temp["cms_demand_match"].sum()
    )

    total = len(temp)

    positives = temp[
        temp["shortage_target"] == 1
    ]

    positive_matches = int(
        positives["cms_demand_match"].sum()
    )

    print(
        f"{year}: "
        f"{matched:,}/{total:,} packages linked to CMS "
        f"({matched / total * 100:.2f}%) | "
        f"shortage cases linked: "
        f"{positive_matches}/{len(positives)}"
    )

In [ ]:
for name in ("pharmaopt_v03_df", "cms_features"):
    globals().pop(name, None)
gc.collect()

## C. Audit and permanently export the historical v0.4 panel
No existing CSV is overwritten without an explicit warning; this pipeline's output filenames are specific to the reproducible rebuild. The original Notebook 06 model definition remains intact.

In [ ]:
# Feature definitions copied from Notebook 06, Cells 29 and 34.
features_v03_numeric = [

    # Product characteristics
    "product_age_days",

    # Recall history
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",
    "days_since_last_recall",
    "has_prior_recall",

    # v0.2 shortage history
    "package_prior_shortages",
    "product_prior_shortages",
    "generic_prior_shortages",
    "labeler_prior_shortages",

    "generic_shortages_prior_1y",
    "labeler_shortages_prior_1y",

    "package_has_shortage_history",
    "product_has_shortage_history",
    "generic_has_shortage_history",
    "labeler_has_shortage_history",

    # NEW v0.3 breadth
    "product_active_packages",
    "generic_active_packages",
    "labeler_active_packages",

    # NEW recency
    "package_days_since_shortage",
    "product_days_since_shortage",
    "generic_days_since_shortage",
    "labeler_days_since_shortage",

    # NEW affected-package breadth
    "product_prior_affected_packages",
    "generic_prior_affected_packages",
    "labeler_prior_affected_packages",

    # NEW recent pressure
    "package_shortages_prior_1y",
    "product_shortages_prior_1y",
    "generic_recent_affected_packages",
    "labeler_recent_affected_packages",

    # NEW normalized pressure
    "product_shortage_pressure",
    "generic_shortage_pressure",
    "labeler_shortage_pressure",

    # NEW sibling signal
    "product_sibling_shortage_packages",
    "generic_sibling_shortage_packages",

    # Log counts
    "log_labeler_prior_shortages",
    "log_generic_prior_shortages"
]

features_v03_cat = [
    "dosage_form",
    "marketing_category"
]

features_v03 = (
    features_v03_numeric
    +
    features_v03_cat
)

features_v04_numeric = features_v03_numeric + [

    # CMS DEMAND LEVEL
    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost",

    # Product-market breadth within CMS
    "cms_brand_count",

    # Demand acceleration
    "cms_claim_growth",
    "cms_fill_growth",

    # Explicitly tell model whether CMS matched
    "cms_demand_match"
]

features_v04_cat = [
    "dosage_form",
    "marketing_category"
]

features_v04 = (
    features_v04_numeric
    +
    features_v04_cat
)
print("Feature definitions loaded successfully.")
print("v0.3 features:", len(features_v03))
print("v0.4 features:", len(features_v04))

In [ ]:
# Key-grain and temporal-label audits.
required = ["package_ndc", "prediction_year", "shortage_target"] + features_v04
missing_columns = sorted(set(required) - set(pharmaopt_v04_df.columns))
if missing_columns:
    raise ValueError(f"Missing original v0.4 model columns: {missing_columns}")

pharmaopt_v04_df["package_ndc"] = (
    pharmaopt_v04_df["package_ndc"].astype("string").str.strip()
)
pharmaopt_v04_df["prediction_year"] = (
    pd.to_numeric(pharmaopt_v04_df["prediction_year"], errors="raise").astype(int)
)

if pharmaopt_v04_df[["package_ndc", "prediction_year"]].duplicated().any():
    raise ValueError("Original panel has duplicate package-year keys; do not join until resolved.")
if pharmaopt_v04_df["package_ndc"].isna().any():
    raise ValueError("Original panel contains null package NDCs.")
if not pharmaopt_v04_df["shortage_target"].isin([0, 1]).all():
    raise ValueError("Shortage targets must be 0/1.")
if sorted(pharmaopt_v04_df["prediction_year"].unique().tolist()) != [2022, 2023, 2024, 2025]:
    raise ValueError("Unexpected prediction years.")

print("PHARMAOPT v0.4 — REBUILT PANEL AUDIT")
print("Rows:", f"{len(pharmaopt_v04_df):,}")
print("Positive shortage labels:", int(pharmaopt_v04_df["shortage_target"].sum()))
print("Feature count:", len(features_v04))
print(pharmaopt_v04_df.groupby("prediction_year")["shortage_target"].agg(["size", "sum"]))

EXPECTED_ROWS = 762_907
EXPECTED_POSITIVES = 426
if len(pharmaopt_v04_df) != EXPECTED_ROWS or int(pharmaopt_v04_df["shortage_target"].sum()) != EXPECTED_POSITIVES:
    print("WARNING: This rebuild differs from the previously recorded 762,907 rows / 426 positives.")
    print("Check source-file and MySQL snapshot versions before interpreting model changes.")

v04_export = RESULTS_DIR / "pharmaopt_v04_model_panel_rebuilt.csv"
if v04_export.exists():
    print("WARNING: Replacing prior generated rebuild export:", v04_export.name)
pharmaopt_v04_df.to_csv(v04_export, index=False, date_format="%Y-%m-%d")

metadata = {
    "source_notebook": "06_pharmaopt_analytics (uploaded source code)",
    "modeling_grain": "package_ndc + prediction_year",
    "prediction_years": [2022, 2023, 2024, 2025],
    "source_table_counts": source_counts,
    "rows": int(len(pharmaopt_v04_df)),
    "shortage_positives": int(pharmaopt_v04_df["shortage_target"].sum()),
    "v04_numeric_features": features_v04_numeric,
    "v04_categorical_features": features_v04_cat,
    "caveats": [
        "Retrospective FDA NDC snapshot; historical cohort survivorship possible",
        "Original v0.4 recall features use initiation dates, not verified publication dates",
        "CMS Part D prior-year utilization may not have been released by January 1",
        "2025 is a development holdout already inspected in prior iterations",
    ],
}
metadata_path = RESULTS_DIR / "pharmaopt_v05_pipeline_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", v04_export)
print("Saved:", metadata_path)

## D. Join the historical qualitative recall-risk features
Notebook 10 produced **10,542 sparse package-year rows** with three-year recall history. This is **not** the full modeling cohort. We use a **left one-to-one join** to preserve all package-years, and explicitly record whether a package-year had a linked recall-history record. A missing linked record is not proof that the drug never had a recall.

In [ ]:
qualitative_file = required_files["Temporal risk features"]
qualitative_features = pd.read_csv(
    qualitative_file,
    dtype={"linked_package_ndc": "string"},
    low_memory=False,
)
qualitative_features = qualitative_features.rename(
    columns={"linked_package_ndc": "package_ndc", "year": "prediction_year"}
)
qualitative_features["package_ndc"] = (
    qualitative_features["package_ndc"].astype("string").str.strip()
)
qualitative_features["prediction_year"] = pd.to_numeric(
    qualitative_features["prediction_year"], errors="raise"
).astype(int)

qualitative_feature_cols = [
    c for c in qualitative_features.columns
    if c.startswith("prior_") and (c.endswith("_3y") or c.endswith("_1y"))
]
assert len(qualitative_feature_cols) == 16, (
    f"Expected 16 temporal count features (3 counts + 13 categories); got {len(qualitative_feature_cols)}"
)
assert not qualitative_features[["package_ndc", "prediction_year"]].duplicated().any()
assert not qualitative_features[["package_ndc", "prediction_year"]].isna().any().any()

for col in qualitative_feature_cols:
    qualitative_features[col] = pd.to_numeric(qualitative_features[col], errors="raise")
    if qualitative_features[col].isna().any() or (qualitative_features[col] < 0).any():
        raise ValueError(f"Invalid or missing historical risk counts: {col}")

assert (
    qualitative_features["prior_recall_count_1y"]
    <= qualitative_features["prior_recall_count_3y"]
).all()

baseline_keys = pharmaopt_v04_df[
    ["package_ndc", "prediction_year", "shortage_target"]
].reset_index(drop=True)

pharmaopt_v05_df = pharmaopt_v04_df.merge(
    qualitative_features,
    on=["package_ndc", "prediction_year"],
    how="left", validate="one_to_one", indicator=True,
    sort=False,
)
pharmaopt_v05_df["linked_recall_history_record_3y"] = (
    pharmaopt_v05_df["_merge"].eq("both").astype(int)
)
pharmaopt_v05_df = pharmaopt_v05_df.drop(columns=["_merge"])
pharmaopt_v05_df[qualitative_feature_cols] = (
    pharmaopt_v05_df[qualitative_feature_cols].fillna(0).astype("int32")
)

pd.testing.assert_frame_equal(
    baseline_keys,
    pharmaopt_v05_df[["package_ndc", "prediction_year", "shortage_target"]].reset_index(drop=True),
    check_dtype=True,
)
assert len(pharmaopt_v05_df) == len(pharmaopt_v04_df)
assert not pharmaopt_v05_df[["package_ndc", "prediction_year"]].duplicated().any()
assert not pharmaopt_v05_df[qualitative_feature_cols].isna().any().any()

print("PHARMAOPT v0.5 — INTEGRATION AUDIT")
print("v0.4 rows:", f"{len(pharmaopt_v04_df):,}")
print("v0.5 rows:", f"{len(pharmaopt_v05_df):,}")
print("Original targets preserved:", True)
print("Temporal qualitative features:", len(qualitative_feature_cols))
print("Historical recall-feature matches:", int(pharmaopt_v05_df["linked_recall_history_record_3y"].sum()))
print("Historical feature coverage by year:")
print(pharmaopt_v05_df.groupby("prediction_year")["linked_recall_history_record_3y"].agg(["size", "sum", "mean"]))

if pharmaopt_v05_df["linked_recall_history_record_3y"].sum() == 0:
    raise ValueError("No historical risk features matched. Investigate NDC formatting before export.")

v05_export = RESULTS_DIR / "pharmaopt_integrated_package_year_v05.csv"
if v05_export.exists():
    print("WARNING: Replacing prior generated integration export:", v05_export.name)
pharmaopt_v05_df.to_csv(v05_export, index=False, date_format="%Y-%m-%d")
print("Saved:", v05_export)

# v0.5 contains every v0.4 field. Releasing the duplicate table reduces memory use.
del pharmaopt_v04_df
gc.collect()

In [ ]:

# PHARMAOPT — INTEGRATION RECOVERY CHECK

print("v0.4 in memory:", "pharmaopt_v04_df" in globals())
print("v0.5 in memory:", "pharmaopt_v05_df" in globals())

v05_file = OUTPUT_DIR / "pharmaopt_integrated_package_year_v05.csv"

print("v0.5 CSV saved:", v05_file.is_file())

if v05_file.is_file():
    print("File size (MB):", round(v05_file.stat().st_size / 1_000_000, 2))
    print("Location:", v05_file)


In [ ]:

# ============================================================
# PHARMAOPT — v0.5 FINAL DATASET AUDIT
# ============================================================

df = pharmaopt_v05_df

print("=" * 65)
print("PHARMAOPT v0.5 — FINAL DATASET AUDIT")
print("=" * 65)

print("Total observations:", f"{len(df):,}")
print("Total columns:", len(df.columns))
print("Total shortage targets:", f"{df['shortage_target'].sum():,}")

duplicates = df.duplicated(
    subset=["package_ndc", "prediction_year"]
).sum()

print("Duplicate package-year records:", duplicates)

print("\nYEARLY MODELING SUMMARY")
print("-" * 65)

summary = df.groupby("prediction_year").agg(
    observations=("package_ndc", "size"),
    shortages=("shortage_target", "sum"),
    linked_recall_history=("linked_recall_history_record_3y", "sum")
)

print(summary.to_string())

print("\nQUALITATIVE INTEGRATION")
print("-" * 65)

print(
    "Package-years with linked recall history:",
    f"{df['linked_recall_history_record_3y'].sum():,}"
)

print(
    "Package-years without linked recall history:",
    f"{(df['linked_recall_history_record_3y'] == 0).sum():,}"
)

print(
    "Linked-history coverage:",
    f"{df['linked_recall_history_record_3y'].mean():.2%}"
)

print("\nEXPECTED BASELINE CHECK")
print("-" * 65)

print("Expected observations: 762,907")
print("Actual observations:", f"{len(df):,}")

print("Expected shortage targets: 426")
print("Actual shortage targets:", f"{df['shortage_target'].sum():,}")

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


## E. Tableau exports: yearly summary and long-form risk categories
**Important:** A recall can have multiple risk labels and can affect multiple packages. Summing risk-category counts does **not** equal the number of distinct recall events. For event counts in Tableau, use `COUNTD([recall_number])` on the recall detail table; for package counts, use `COUNTD([package_ndc])`.

In [ ]:
# Package-year summary: exactly one row per prediction year.
yearly_summary = (
    pharmaopt_v05_df.groupby("prediction_year", as_index=False)
    .agg(
        package_years=("package_ndc", "size"),
        distinct_packages=("package_ndc", "nunique"),
        shortage_positive_packages=("shortage_target", "sum"),
        cms_matched_packages=("cms_demand_match", "sum"),
        packages_with_linked_recall_history_3y=("linked_recall_history_record_3y", "sum"),
    )
)
yearly_summary["shortage_rate"] = (
    yearly_summary["shortage_positive_packages"] / yearly_summary["package_years"]
)
yearly_summary["cms_match_rate"] = (
    yearly_summary["cms_matched_packages"] / yearly_summary["package_years"]
)
yearly_summary["linked_recall_history_3y_rate"] = (
    yearly_summary["packages_with_linked_recall_history_3y"] / yearly_summary["package_years"]
)

summary_export = RESULTS_DIR / "pharmaopt_tableau_yearly_summary_v05.csv"
yearly_summary.to_csv(summary_export, index=False)
print("Saved:", summary_export)
display(yearly_summary)

# Sparse, long-form category counts: only positive category counts are retained.
risk_count_cols = [
    c for c in qualitative_feature_cols
    if c.startswith("prior_") and c.endswith("_risk_count_3y")
]
assert len(risk_count_cols) == 13

model_lookup = pharmaopt_v05_df[
    ["package_ndc", "prediction_year", "generic_name", "brand_name", "labeler_name", "shortage_target"]
].copy()

risk_tableau = qualitative_features[
    ["package_ndc", "prediction_year"] + risk_count_cols
].merge(
    model_lookup,
    on=["package_ndc", "prediction_year"],
    how="left", validate="one_to_one", indicator=True,
)
risk_tableau["in_model_cohort"] = risk_tableau["_merge"].eq("both").astype(int)
risk_tableau = risk_tableau.drop(columns="_merge")
risk_tableau = risk_tableau.melt(
    id_vars=["package_ndc", "prediction_year", "generic_name", "brand_name", "labeler_name", "shortage_target", "in_model_cohort"],
    value_vars=risk_count_cols,
    var_name="risk_category", value_name="recall_category_count_prior_3y",
)
risk_tableau = risk_tableau.loc[
    risk_tableau["recall_category_count_prior_3y"] > 0
].copy()
risk_tableau["risk_category"] = (
    risk_tableau["risk_category"]
    .str.removeprefix("prior_")
    .str.removesuffix("_count_3y")
)

risk_tableau_export = RESULTS_DIR / "pharmaopt_tableau_risk_categories_v05.csv"
risk_tableau.to_csv(risk_tableau_export, index=False)
print("Saved:", risk_tableau_export)
print("Category observations:", len(risk_tableau))
display(risk_tableau.head())

### Recall-level detail for Tableau
This table is **one row per linked recall × package**, not one row per unique recall. It includes reported dates, classification, reason text, and the rule-based category flags. The additional long-form table contains only positive category labels for easier filtering.

In [ ]:
recall_detail_sql = """
SELECT
    re.recall_number,
    re.report_date,
    re.recall_initiation_date,
    re.classification,
    re.status,
    re.recalling_firm,
    re.reason_for_recall,
    rp.source_package_ndc,
    rp.linked_package_ndc AS package_ndc
FROM recall_event AS re
JOIN recall_package AS rp ON re.recall_number = rp.recall_number
WHERE rp.linked_package_ndc IS NOT NULL
  AND TRIM(rp.linked_package_ndc) <> ''
"""
recall_detail = query_df(recall_detail_sql)
recall_detail["recall_number"] = recall_detail["recall_number"].astype("string").str.strip()
recall_detail["package_ndc"] = recall_detail["package_ndc"].astype("string").str.strip()
recall_detail["report_date"] = pd.to_datetime(recall_detail["report_date"], errors="coerce")
recall_detail["recall_initiation_date"] = pd.to_datetime(recall_detail["recall_initiation_date"], errors="coerce")

if recall_detail[["recall_number", "package_ndc"]].duplicated().any():
    raise ValueError("Duplicate recall/package keys in recall detail. Audit before Tableau export.")

risk_labels = pd.read_csv(
    required_files["Recall classifications"],
    dtype={"recall_number": "string"},
    keep_default_na=False,
    low_memory=False,
)
risk_labels["recall_number"] = risk_labels["recall_number"].astype("string").str.strip()
assert not risk_labels["recall_number"].duplicated().any()

risk_categories = [
    c.removeprefix("prior_").removesuffix("_count_3y")
    for c in risk_count_cols
]
label_cols = ["recall_number", "risk_signal_count_v2", "unclassified_risk_v2"] + risk_categories
missing_labels = sorted(set(label_cols) - set(risk_labels.columns))
if missing_labels:
    raise ValueError(f"Risk classification file lacks columns: {missing_labels}")

recall_detail = recall_detail.merge(
    risk_labels[label_cols], on="recall_number", how="left",
    validate="many_to_one", indicator=True,
)
recall_detail["risk_classification_joined"] = recall_detail["_merge"].eq("both").astype(int)
recall_detail = recall_detail.drop(columns="_merge")
recall_detail["recall_report_year"] = recall_detail["report_date"].dt.year

print("Recall-package detail rows:", f"{len(recall_detail):,}")
print("Distinct recall events:", recall_detail["recall_number"].nunique())
print("Distinct linked packages:", recall_detail["package_ndc"].nunique())
print("Matched risk classifications:", int(recall_detail["risk_classification_joined"].sum()))
print("Unmatched classifications:", int((recall_detail["risk_classification_joined"] == 0).sum()))

recall_export = RESULTS_DIR / "pharmaopt_recall_risk_detail_v05.csv"
recall_detail.to_csv(recall_export, index=False, date_format="%Y-%m-%d")
print("Saved:", recall_export)

category_detail = recall_detail.melt(
    id_vars=["recall_number", "package_ndc", "report_date", "recall_report_year", "classification", "recalling_firm"],
    value_vars=risk_categories + ["unclassified_risk_v2"],
    var_name="risk_category", value_name="category_flag",
)
category_detail["category_flag"] = pd.to_numeric(category_detail["category_flag"], errors="coerce")
category_detail = category_detail.loc[category_detail["category_flag"] == 1].copy()
category_detail["risk_category"] = category_detail["risk_category"].replace(
    {"unclassified_risk_v2": "unclassified_risk"}
)
category_detail = category_detail.drop(columns="category_flag")
category_detail_export = RESULTS_DIR / "pharmaopt_tableau_recall_categories_v05.csv"
category_detail.to_csv(category_detail_export, index=False, date_format="%Y-%m-%d")
print("Saved:", category_detail_export)
print("NOTE: A recall may appear in multiple category rows and across packages.")

# All MySQL reads are now complete.
connection.close()
print("MySQL connection closed; no database writes were made.")

## F. Optional: train v0.4 and v0.5 on the same chronological split
**Run only after the integration audit looks correct.** Both models use the original Notebook 06 preprocessing and balanced logistic regression. Train on **2022–2024** and evaluate on the already-inspected **2025 development holdout**. These scores are for **ranking**, not calibrated shortage probabilities. A positive change in a metric would be exploratory, not external validation.

In [ ]:

# ============================================================
# PHARMAOPT — SECTION F
# MACHINE LEARNING COMPARISON: v0.4 VS v0.5
# WITH LIVE TRAINING STATUS AND TIMING
# ============================================================

import gc
import time
import threading
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score


# ------------------------------------------------------------
# 1. IMMEDIATE CONFIRMATION
# ------------------------------------------------------------

display(HTML("""
<div style="
    background:#e7f6ec;
    color:#14532d;
    padding:18px;
    border:2px solid #22a06b;
    border-radius:10px;
    font-size:18px;
    font-weight:bold;
    margin:10px 0;">
    PHARMAOPT SECTION F STARTED
    <div style="font-size:13px; font-weight:normal; margin-top:6px;">
        Python is executing the machine-learning pipeline.
        Training status messages will appear below.
    </div>
</div>
"""))


def log(message):
    current_time = datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] {message}", flush=True)


log("Section F started successfully.")
log("Beginning dataset and feature checks...")


# ------------------------------------------------------------
# 2. LOCATE INTEGRATED DATASET
# ------------------------------------------------------------

v05_export = RESULTS_DIR / "pharmaopt_integrated_package_year_v05.csv"

if not v05_export.is_file():
    raise FileNotFoundError(
        f"Cannot find integrated dataset: {v05_export}"
    )

log("Integrated v0.5 CSV found.")


# ------------------------------------------------------------
# 3. VERIFY MODEL FEATURE DEFINITIONS
# ------------------------------------------------------------

required_variables = [
    "features_v04_numeric",
    "features_v04_cat",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Missing original feature definitions: "
        + ", ".join(missing_variables)
        + ". Run the earlier feature-definition cell first."
    )

available_columns = pd.read_csv(
    v05_export,
    nrows=0
).columns.tolist()

qualitative_feature_cols = [
    c for c in available_columns
    if c.startswith("prior_")
    and (c.endswith("_3y") or c.endswith("_1y"))
]

if len(qualitative_feature_cols) != 16:
    raise ValueError(
        f"Expected 16 historical risk features; "
        f"found {len(qualitative_feature_cols)}."
    )


# ------------------------------------------------------------
# 4. DEFINE v0.4 AND v0.5 MODEL FEATURES
# ------------------------------------------------------------

model_features_v04 = list(dict.fromkeys(
    list(features_v04_numeric)
    + list(features_v04_cat)
))

model_numeric_v05 = list(dict.fromkeys(
    list(features_v04_numeric)
    + qualitative_feature_cols
    + ["linked_recall_history_record_3y"]
))

model_cat_v05 = list(features_v04_cat)

model_features = list(dict.fromkeys(
    model_numeric_v05 + model_cat_v05
))

keys_and_labels = [
    "package_ndc",
    "product_ndc",
    "prediction_year",
    "shortage_target",
    "generic_name",
    "brand_name",
    "labeler_name",
    "cms_demand_match",
    "cms_total_claims",
    "cms_30day_fills",
    "prior_recall_count_3y",
]

keep_columns = list(dict.fromkeys(
    keys_and_labels + model_features
))

missing_columns = [
    c for c in keep_columns
    if c not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Missing required dataset columns: {missing_columns}"
    )

log(f"v0.4 features: {len(model_features_v04)}")
log(f"v0.5 features: {len(model_features)}")
log("Feature definitions verified.")


# ------------------------------------------------------------
# 5. LOAD OR REUSE TRAINING PANEL
# ------------------------------------------------------------

if (
    "training_panel" in globals()
    and isinstance(training_panel, pd.DataFrame)
    and set(keep_columns).issubset(training_panel.columns)
):
    log("Reusing training panel already in memory.")
    training_panel = training_panel[keep_columns]

elif (
    "pharmaopt_v05_df" in globals()
    and isinstance(pharmaopt_v05_df, pd.DataFrame)
):
    log("Preparing training panel from v0.5 in memory.")

    training_panel = pharmaopt_v05_df[
        keep_columns
    ].copy()

else:
    log("Loading selected columns from the saved CSV.")
    log("This may take a minute.")

    training_panel = pd.read_csv(
        v05_export,
        usecols=keep_columns,
        dtype={
            "package_ndc": "string",
            "product_ndc": "string",
        },
        low_memory=False,
    )

    training_panel = training_panel[keep_columns]

log(f"Training panel loaded: {len(training_panel):,} rows.")


# ------------------------------------------------------------
# 6. DATA INTEGRITY CHECKS
# ------------------------------------------------------------

if len(training_panel) != 762907:
    raise ValueError("Unexpected modeling observation count.")

if int(training_panel["shortage_target"].sum()) != 426:
    raise ValueError("Unexpected shortage target count.")

duplicates = training_panel.duplicated(
    subset=["package_ndc", "prediction_year"]
).sum()

if duplicates != 0:
    raise ValueError(
        f"Found {duplicates} duplicate package-year records."
    )

log("Dataset integrity checks passed.")


# ------------------------------------------------------------
# 7. RELEASE UNNECESSARY LARGE TABLES
# ------------------------------------------------------------

for name in (
    "pharmaopt_v05_df",
    "model_lookup",
    "risk_tableau",
    "recall_detail",
    "category_detail",
):
    globals().pop(name, None)

gc.collect()

log("Memory cleanup completed.")


# ------------------------------------------------------------
# 8. CHRONOLOGICAL TRAIN / EVALUATION SPLIT
# ------------------------------------------------------------

train_mask = training_panel["prediction_year"] < 2025
test_mask = training_panel["prediction_year"] == 2025

X_train_v04 = training_panel.loc[
    train_mask, model_features_v04
]

X_test_v04 = training_panel.loc[
    test_mask, model_features_v04
]

X_train_v05 = training_panel.loc[
    train_mask, model_features
]

X_test_v05 = training_panel.loc[
    test_mask, model_features
]

y_train = training_panel.loc[
    train_mask, "shortage_target"
].astype(int)

y_test = training_panel.loc[
    test_mask, "shortage_target"
].astype(int)

if len(y_train) != 541240 or int(y_train.sum()) != 330:
    raise ValueError("Unexpected training split.")

if len(y_test) != 221667 or int(y_test.sum()) != 96:
    raise ValueError("Unexpected evaluation split.")

log(
    f"Training: {len(y_train):,} rows, "
    f"{int(y_train.sum())} shortages."
)

log(
    f"2025 evaluation: {len(y_test):,} rows, "
    f"{int(y_test.sum())} shortages."
)

log("Chronological split verified.")


# ------------------------------------------------------------
# 9. MODEL PIPELINE
# ------------------------------------------------------------

def make_pharmaopt_logistic(
    numeric_features,
    categorical_features
):

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipe,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipe,
                categorical_features
            ),
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2500,
                solver="liblinear"
            )
        ),
    ])


# ------------------------------------------------------------
# 10. TRAINING FUNCTION WITH HEARTBEAT
# ------------------------------------------------------------

def train_with_progress(model, X, y, model_name):

    log("=" * 55)
    log(f"STARTING {model_name} TRAINING")
    log("=" * 55)

    start_time = time.perf_counter()
    stop_event = threading.Event()

    def heartbeat():
        while not stop_event.wait(30):
            elapsed = time.perf_counter() - start_time

            log(
                f"{model_name} STILL TRAINING "
                f"| {elapsed / 60:.1f} minutes elapsed"
            )

    progress_thread = threading.Thread(
        target=heartbeat,
        daemon=True
    )

    progress_thread.start()

    try:
        model.fit(X, y)

    finally:
        stop_event.set()
        progress_thread.join(timeout=2)

    elapsed = time.perf_counter() - start_time

    log(
        f"{model_name} TRAINING COMPLETED "
        f"in {elapsed / 60:.2f} minutes."
    )

    return model


# ------------------------------------------------------------
# 11. TRAIN v0.4 BASELINE
# ------------------------------------------------------------

model_v04 = make_pharmaopt_logistic(
    features_v04_numeric,
    features_v04_cat
)

model_v04 = train_with_progress(
    model_v04,
    X_train_v04,
    y_train,
    "PHARMAOPT v0.4"
)

log("Generating v0.4 evaluation scores.")

score_v04 = model_v04.predict_proba(
    X_test_v04
)[:, 1]

log("v0.4 evaluation scores generated.")


# ------------------------------------------------------------
# 12. TRAIN v0.5 QUALITATIVE RISK MODEL
# ------------------------------------------------------------

model_v05 = make_pharmaopt_logistic(
    model_numeric_v05,
    model_cat_v05
)

model_v05 = train_with_progress(
    model_v05,
    X_train_v05,
    y_train,
    "PHARMAOPT v0.5"
)

log("Generating v0.5 evaluation scores.")

score_v05 = model_v05.predict_proba(
    X_test_v05
)[:, 1]

log("v0.5 evaluation scores generated.")


# ------------------------------------------------------------
# 13. PERFORMANCE METRICS
# ------------------------------------------------------------

log("Calculating model performance metrics.")

pr_auc_v04 = average_precision_score(
    y_test, score_v04
)

pr_auc_v05 = average_precision_score(
    y_test, score_v05
)

roc_auc_v04 = roc_auc_score(
    y_test, score_v04
)

roc_auc_v05 = roc_auc_score(
    y_test, score_v05
)

base_rate = y_test.mean()


def top_500_hits(scores):
    top_indices = np.argsort(-scores)[:500]
    return int(y_test.to_numpy()[top_indices].sum())


top500_v04 = top_500_hits(score_v04)
top500_v05 = top_500_hits(score_v05)


# ------------------------------------------------------------
# 14. DISPLAY FINAL COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PHARMAOPT — v0.4 VS v0.5 RESULTS")
print("=" * 65)

results = pd.DataFrame({
    "Metric": [
        "PR-AUC",
        "ROC-AUC",
        "PR-AUC lift vs random",
        "Shortages found in top 500"
    ],
    "v0.4": [
        round(pr_auc_v04, 6),
        round(roc_auc_v04, 6),
        round(pr_auc_v04 / base_rate, 2),
        top500_v04
    ],
    "v0.5": [
        round(pr_auc_v05, 6),
        round(roc_auc_v05, 6),
        round(pr_auc_v05 / base_rate, 2),
        top500_v05
    ]
})

display(results)

print("\nPR-AUC change:", f"{pr_auc_v05 - pr_auc_v04:+.6f}")
print("ROC-AUC change:", f"{roc_auc_v05 - roc_auc_v04:+.6f}")
print("Top-500 change:", f"{top500_v05 - top500_v04:+d}")

display(HTML("""
<div style="
    background:#e7f6ec;
    color:#14532d;
    padding:18px;
    border:2px solid #22a06b;
    border-radius:10px;
    font-size:18px;
    font-weight:bold;
    margin:10px 0;">
    PHARMAOPT SECTION F COMPLETED
    <div style="font-size:13px; font-weight:normal; margin-top:6px;">
        Both models finished training and their
        2025 evaluation metrics were calculated.
    </div>
</div>
"""))

log("Section F completed successfully.")


In [ ]:

# ============================================================
# PHARMAOPT — SECTION F FINAL EXPORTS
# MODEL COMPARISON + 2025 PACKAGE RISK RANKINGS
# ============================================================

print("=" * 65, flush=True)
print("PHARMAOPT — FINAL EXPORT STARTED", flush=True)
print("=" * 65, flush=True)

import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. OUTPUT DIRECTORY
# ------------------------------------------------------------

# Save this experiment separately from the original v0.4/v0.5 results.
OUTPUT_DIR = RESULTS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nOutput folder:", OUTPUT_DIR, flush=True)


# ------------------------------------------------------------
# 2. CHECK PREVIOUS TRAINING RESULTS
# ------------------------------------------------------------

required = [
    "training_panel",
    "test_mask",
    "y_test",
    "score_v04",
    "score_v05"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing training results: "
        + ", ".join(missing)
        + ". The completed Section F training results "
          "must still be available in memory."
    )

print("Both model score arrays found.", flush=True)


# ------------------------------------------------------------
# 3. PREPARE 2025 EVALUATION DATA
# ------------------------------------------------------------

score_cols = [
    "package_ndc",
    "product_ndc",
    "prediction_year",
    "shortage_target",
    "generic_name",
    "brand_name",
    "labeler_name",
    "cms_demand_match",
    "cms_total_claims",
    "cms_30day_fills",
    "prior_recall_count_3y"
]

score_table = (
    training_panel.loc[test_mask, score_cols]
    .copy()
    .reset_index(drop=True)
)

y = score_table["shortage_target"].to_numpy(dtype=int)

scores04 = np.asarray(score_v04, dtype=float)
scores05 = np.asarray(score_v05, dtype=float)

if not (
    len(y) == len(scores04) == len(scores05) == len(y_test)
):
    raise ValueError("Model scores and evaluation rows do not match.")

if not np.array_equal(
    y,
    np.asarray(y_test, dtype=int)
):
    raise ValueError("Evaluation targets are out of order.")

if not (
    np.isfinite(scores04).all()
    and np.isfinite(scores05).all()
):
    raise ValueError("Model scores contain invalid values.")

if len(y) != 221667 or int(y.sum()) != 96:
    raise ValueError("Unexpected 2025 evaluation dataset.")

if score_table["package_ndc"].isna().any():
    raise ValueError("Missing package NDC identifiers.")

if score_table["package_ndc"].duplicated().any():
    raise ValueError("Duplicate 2025 package identifiers.")

print("\n2025 evaluation verified.", flush=True)
print("Packages:", f"{len(y):,}", flush=True)
print("Shortage positives:", int(y.sum()), flush=True)


# ------------------------------------------------------------
# 4. DEFINE MODEL EVALUATION METRICS
# ------------------------------------------------------------

package_ids = (
    score_table["package_ndc"]
    .astype(str)
    .to_numpy()
)

def calculate_model_metrics(model_name, scores):

    baseline_rate = float(y.mean())

    # Sort by descending score, then package NDC.
    order = np.lexsort((package_ids, -scores))

    pr_auc = float(
        average_precision_score(y, scores)
    )

    roc_auc = float(
        roc_auc_score(y, scores)
    )

    result = {
        "model": model_name,
        "evaluation_year": 2025,
        "evaluation_type":
            "development_holdout_previously_inspected",
        "evaluated_packages": len(y),
        "shortage_positives": int(y.sum()),
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "baseline_positive_rate": baseline_rate,
        "PR_AUC_Lift": pr_auc / baseline_rate
    }

    for k in (
        100, 500, 1000, 2500,
        5000, 10000, 25000
    ):
        result[f"Top_{k}_Captured"] = int(
            y[order[:k]].sum()
        )

    return result


# ------------------------------------------------------------
# 5. CREATE MODEL COMPARISON
# ------------------------------------------------------------

print("\nCalculating model comparison...", flush=True)

model_comparison_v05 = pd.DataFrame([
    calculate_model_metrics(
        "v0.4_rebuilt",
        scores04
    ),
    calculate_model_metrics(
        "v0.5_qualitative",
        scores05
    )
])

comparison_export = (
    OUTPUT_DIR / "pharmaopt_model_comparison_v05.csv"
)

model_comparison_v05.to_csv(
    comparison_export,
    index=False
)

print("\nSAVED MODEL COMPARISON:", flush=True)
print(comparison_export, flush=True)

display(model_comparison_v05)


# ------------------------------------------------------------
# 6. GENERATE PACKAGE-LEVEL RISK RANKINGS
# ------------------------------------------------------------

print("\nGenerating package risk rankings...", flush=True)

for version, scores in (
    ("v04", scores04),
    ("v05", scores05)
):

    score_table[f"risk_score_{version}"] = scores

    order = np.lexsort(
        (package_ids, -scores)
    )

    ranks = np.empty(
        len(scores),
        dtype=np.int32
    )

    ranks[order] = np.arange(
        1,
        len(scores) + 1,
        dtype=np.int32
    )

    score_table[f"risk_rank_{version}"] = ranks

score_table["rank_improvement_v05"] = (
    score_table["risk_rank_v04"]
    - score_table["risk_rank_v05"]
)

print("Package rankings completed.", flush=True)


# ------------------------------------------------------------
# 7. EXPORT 2025 PACKAGE SCORES
# ------------------------------------------------------------

score_export = (
    OUTPUT_DIR / "pharmaopt_2025_model_scores_v05.csv"
)

score_table.to_csv(
    score_export,
    index=False
)

print("\nSAVED PACKAGE SCORES:", flush=True)
print(score_export, flush=True)
print(
    "Rows exported:",
    f"{len(score_table):,}",
    flush=True
)


# ------------------------------------------------------------
# 8. PRINT FINAL PERFORMANCE SUMMARY
# ------------------------------------------------------------

v04 = model_comparison_v05.iloc[0]
v05 = model_comparison_v05.iloc[1]

print("\n" + "=" * 65)
print("PHARMAOPT — FINAL MODEL PERFORMANCE")
print("=" * 65)

print(
    f"v0.4 PR-AUC: {v04['PR_AUC']:.6f}"
)
print(
    f"v0.5 PR-AUC: {v05['PR_AUC']:.6f}"
)

print(
    "PR-AUC change:",
    f"{v05['PR_AUC'] - v04['PR_AUC']:+.6f}"
)

print(
    f"\nv0.4 ROC-AUC: {v04['ROC_AUC']:.6f}"
)
print(
    f"v0.5 ROC-AUC: {v05['ROC_AUC']:.6f}"
)

print(
    "ROC-AUC change:",
    f"{v05['ROC_AUC'] - v04['ROC_AUC']:+.6f}"
)

print(
    "\nTop-500 shortages detected:",
    f"{int(v04['Top_500_Captured'])} vs "
    f"{int(v05['Top_500_Captured'])}"
)


# ------------------------------------------------------------
# 9. VERIFY EXPORTED FILES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL EXPORT VERIFICATION")
print("=" * 65)

for path in (
    comparison_export,
    score_export
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Export failed: {path}"
        )

    print(
        "VERIFIED:",
        path.name,
        f"({path.stat().st_size / (1024**2):.2f} MB)"
    )

print("\n" + "=" * 65)
print("PHARMAOPT — FINAL EXPORT COMPLETED")
print("=" * 65)

print(
    "Both files saved successfully. "
    "Results are retrospective development analyses, "
    "not validated future shortage probabilities."
)


## Tableau usage and data dictionary

| File | Grain | Tableau use |
|---|---|---|
| `pharmaopt_integrated_package_year_v05.csv` | package NDC × prediction year | Full FDA/CMS/qualitative analytics, cohort filters, feature distributions |
| `pharmaopt_tableau_yearly_summary_v05.csv` | prediction year | Yearly shortage rates, CMS coverage, historical recall feature coverage |
| `pharmaopt_tableau_risk_categories_v05.csv` | package NDC × prediction year × *positive risk category* | Historical category counts by year, drug, or labeler; **counts overlap across categories** |
| `pharmaopt_recall_risk_detail_v05.csv` | linked recall × package NDC | Recall date timeline, classification, reason text, unique event counts with `COUNTD` |
| `pharmaopt_tableau_recall_categories_v05.csv` | linked recall × package NDC × positive category | Historical recall categories and trends; **multi-label, so counts overlap** |
| `pharmaopt_2025_model_scores_v05.csv` | 2025 package NDC | v0.4 vs v0.5 ranking scatterplots and top-K capture (after optional training) |
| `pharmaopt_model_comparison_v05.csv` | model version | PR-AUC, ROC-AUC, top-K comparison (after optional training) |

**Model features:** `prior_recall_count_3y`, `prior_recall_count_1y`, `prior_unclassified_count_3y`, the 13 `prior_*_risk_count_3y` columns, and `linked_recall_history_record_3y`.

**Interpretation:** `linked_recall_history_record_3y = 0` means *no linked recall-history record found in this three-year feature table*; it does not guarantee the package had no recalls. The category labels are **rule-based coverage**, not independently validated ground truth.

**Reproducibility:** The code uses your local source snapshots. Do not commit passwords, raw source ZIPs, large generated datasets, or personally identifying files to a public GitHub repository. Commit the notebook, documentation, and small non-sensitive samples instead.

In [ ]:
%pip install mysql-connector-python

In [ ]:
import pandas as pd

columns = query_df("""
    SHOW COLUMNS FROM shortage_event;
""")

display(columns)

In [ ]:
status_counts = query_df("""
    SELECT
        COALESCE(status, '(NULL)') AS status,
        COUNT(*) AS total_records,
        COUNT(DISTINCT linked_package_ndc) AS linked_packages
    FROM shortage_event
    GROUP BY status
    ORDER BY total_records DESC;
""")

display(status_counts)

In [ ]:
status_by_year = query_df("""
    SELECT
        YEAR(initial_posting_date) AS event_year,
        status,
        COUNT(*) AS records,
        COUNT(DISTINCT linked_package_ndc) AS linked_packages
    FROM shortage_event
    WHERE initial_posting_date >= '2022-01-01'
      AND initial_posting_date < '2026-01-01'
    GROUP BY YEAR(initial_posting_date), status
    ORDER BY event_year, status;
""")

display(status_by_year)

In [ ]:
display(query_df("""
    SELECT
        status,
        availability,
        update_type,
        COUNT(*) AS records
    FROM shortage_event
    WHERE initial_posting_date >= '2025-01-01'
      AND initial_posting_date < '2026-01-01'
    GROUP BY status, availability, update_type
    ORDER BY records DESC;
"""))